In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 6)

filename = "data-1.csv"

df = pd.read_csv(filename, header=None)
df.columns = ["x1", "x2", "y"]

x = df[["x1", "x2"]].values
y = df["y"].values.astype(int)

os.makedirs("plots", exist_ok=True)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def plot_points(ax, x, y):
    for label, color, name in [(1, "blue", "Class 1"), (0, "orange", "Class 0")]:
        mask = y == label
        ax.scatter(x[mask, 0], x[mask, 1], c=color, label=name, edgecolors="k", s=40)

def plot_line(ax, w, b, x_vals, color, linestyle="-", linewidth=2, alpha=1.0, label=None):
    if abs(w[1]) < 1e-10:
        return
    y_vals = -(w[0] * x_vals + b) / w[1]
    ax.plot(x_vals, y_vals, color=color, linestyle=linestyle, linewidth=linewidth, alpha=alpha, label=label)

fig, ax = plt.subplots()
plot_points(ax, x, y)
ax.set_title("Plain Data")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("plots/plain_data.png", dpi=200)
plt.show()

def Part_1(x, y, lr=0.1, max_iterations=500, seed=42):
    np.random.seed(seed)

    w = np.random.randn(x.shape[1]) * 0.01
    b = np.random.randn() * 0.01

    boundary_history = [(w.copy(), b)]
    total_points = x.shape[0]

    for iteration in range(1, max_iterations + 1):
        errors = 0

        for i in range(total_points):
            z = np.dot(w, x[i]) + b
            y_hat = 1 if z >= 0 else 0

            if y_hat != y[i]:
                if y_hat == 0:
                    b += lr
                    w += lr * x[i]
                else:
                    b -= lr
                    w -= lr * x[i]
                errors += 1

        boundary_history.append((w.copy(), b))

        if errors == 0:
            break

    return w, b, boundary_history, iteration

learning_rates = [0.01, 0.1, 1.0]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x_min, x_max = x[:, 0].min() - 0.1, x[:, 0].max() + 0.1
y_min, y_max = x[:, 1].min() - 0.1, x[:, 1].max() + 0.1
x_vals = np.linspace(x_min, x_max, 200)

for ax, lr in zip(axes, learning_rates):
    w, b, boundary_history, n_iter = Part_1(x, y, lr=lr, max_iterations=65, seed=42)

    plot_points(ax, x, y)

    start_w, start_b = boundary_history[0]
    plot_line(ax, start_w, start_b, x_vals, color="red", linewidth=2, label="Initial")

    first_mid = True
    for w_step, b_step in boundary_history[1:-1]:
        label = "Iterations" if first_mid else None
        plot_line(ax, w_step, b_step, x_vals, color="green", linestyle="--", linewidth=0.8, alpha=0.5, label=label
        )
        first_mid = False

    final_w, final_b = boundary_history[-1]
    plot_line(ax, final_w, final_b, x_vals, color="black", linewidth=2, label="Final")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"rate={lr} | passes={n_iter}")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plots/part1.png", dpi=200)
plt.show()

def Part_2(x, y, lr=0.1, epochs=65, seed=42):
    np.random.seed(seed)
    w = np.random.randn(2) * 0.01
    b = np.random.randn() * 0.01

    boundary_history = [(w.copy(), b)]
    loss_epochs = []
    losses = []

    for epoch in range(1, epochs + 1):
        for i in range(len(x)):
            z = np.dot(w, x[i]) + b
            y_hat = sigmoid(z)

            error = y_hat - y[i]
            w -= lr * error * x[i]
            b -= lr * error

        boundary_history.append((w.copy(), b))

        if epoch % 10 == 0:
            probs = sigmoid(x @ w + b)
            loss = log_loss(y, probs)
            loss_epochs.append(epoch)
            losses.append(loss)

    return w, b, boundary_history, loss_epochs, losses

w2, b2, history2, loss_epochs2, losses2 = Part_2(x, y)

fig, ax = plt.subplots()
plot_points(ax, x, y)

x_vals = np.linspace(x[:, 0].min() - 0.1, x[:, 0].max() + 0.1, 200)

w0, b0 = history2[0]
plot_line(ax, w0, b0, x_vals, "red", label="Initial")

for j in range(1, len(history2) - 1):
    wj, bj = history2[j]
    lbl = "Intermediate" if j == 1 else None
    plot_line(ax, wj, bj, x_vals, "green", linestyle="--", alpha=0.5, label=lbl)

wf, bf = history2[-1]
plot_line(ax, wf, bf, x_vals, "black", label="Final")

ax.set_xlim(x[:, 0].min() - 0.1, x[:, 0].max() + 0.1)
ax.set_ylim(x[:, 1].min() - 0.1, x[:, 1].max() + 0.1)
ax.set_title("Part 2 Boundary")
ax.legend()

plt.savefig("plots/part2_boundary.png", dpi=150)
plt.show()

plt.figure()
plt.plot(loss_epochs2, losses2, marker="o")
plt.xlabel("Epochs")
plt.ylabel("Log Loss")
plt.title("Part 2 Error")
plt.grid()
plt.savefig("plots/part2_error.png", dpi=150)
plt.show()

ModuleNotFoundError: No module named 'numpy'